# 10-Latent Recovery With Internal `laplace_em`

This notebook builds a deliberately well-identified diagonal 10-latent state-space model,
simulates synthetic data, and fits it with `fit(..., method="laplace_em")` through the
non-Kalman path.

- `SSMModel(..., likelihood="particle")` forces the internal Laplace/IEKS backend instead of the exact Kalman backend.
- `USE_INTERVAL_SUPPORT = True` additionally routes through the support-aware branch, so the outer optimizer is the new exact-gradient `L-BFGS-B` path.
- The model is intentionally simple: fixed identity loadings, diagonal drift, diagonal diffusion, and diagonal observation noise.


In [ ]:
from pathlib import Path
import resource
import sys
import time

ROOT = Path.cwd()
if (ROOT / "src").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError(
        "Run this notebook from apps/data-pipeline or apps/data-pipeline/notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import jax.numpy as jnp
import jax.random as random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from causal_ssm_agent.models.ssm import (
    SSMModel,
    SSMSpec,
    fit,
    full_diagonal_mask,
    zero_diagonal_mask,
    zero_loading_mask,
    zero_square_mask,
    zero_vector_mask,
)
from causal_ssm_agent.models.ssm.diagnostics import simulate_ssm
from causal_ssm_agent.models.ssm_observation_metadata import ObservationSupportRuntime

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
np.set_printoptions(precision=3, suppress=True)


In [ ]:
N_LATENT = 10
N_MANIFEST = 10
T = 40
NUM_SAMPLES = 150
N_IEKS_ITERS = 5
MAXITER = 60
TOL = 1e-4
SEED = 0
USE_INTERVAL_SUPPORT = True

assert N_LATENT == N_MANIFEST

true_drift_diag = -jnp.linspace(0.18, 0.45, N_LATENT, dtype=jnp.float32)
true_diff_diag = jnp.linspace(0.10, 0.18, N_LATENT, dtype=jnp.float32)
true_obs_sd = jnp.linspace(0.08, 0.14, N_MANIFEST, dtype=jnp.float32)
true_t0_sd = jnp.linspace(0.20, 0.32, N_LATENT, dtype=jnp.float32)


def make_diagonal_spec(n_latent: int, n_manifest: int, lambda_mat: jnp.ndarray) -> SSMSpec:
    return SSMSpec(
        n_latent=n_latent,
        n_manifest=n_manifest,
        drift_diag_mask=full_diagonal_mask(n_latent),
        drift_offdiag_mask=zero_square_mask(n_latent),
        drift=jnp.zeros((n_latent, n_latent), dtype=jnp.float32),
        cint_mask=zero_vector_mask(n_latent),
        cint=jnp.zeros(n_latent, dtype=jnp.float32),
        lambda_mask=zero_loading_mask(n_manifest, n_latent),
        lambda_mat=lambda_mat,
        diffusion_chol_mask=np.diag(full_diagonal_mask(n_latent)),
        diffusion_chol=jnp.eye(n_latent, dtype=jnp.float32),
        manifest_means_mask=zero_vector_mask(n_manifest),
        manifest_means=jnp.zeros(n_manifest, dtype=jnp.float32),
        manifest_chol_diag_mask=full_diagonal_mask(n_manifest),
        manifest_chol=jnp.zeros((n_manifest, n_manifest), dtype=jnp.float32),
        t0_means_mask=zero_vector_mask(n_latent),
        t0_means=jnp.zeros(n_latent, dtype=jnp.float32),
        t0_chol_diag_mask=zero_diagonal_mask(n_latent),
        t0_correlation_mask=zero_square_mask(n_latent),
        t0_chol=jnp.diag(true_t0_sd),
        latent_names=[f"x{i}" for i in range(n_latent)],
        manifest_names=[f"y{i}" for i in range(n_manifest)],
    )


def build_interval_mean_support(times: jnp.ndarray, manifest_names: list[str]) -> ObservationSupportRuntime:
    times_np = np.asarray(times, dtype=np.float64)
    n_time = times_np.shape[0]
    n_manifest = len(manifest_names)

    support_start = np.full((n_time, n_manifest), np.nan, dtype=np.float64)
    support_end = np.full((n_time, n_manifest), np.nan, dtype=np.float64)
    prev_coeffs = np.zeros((n_time, n_manifest, 1), dtype=np.float64)
    curr_coeffs = np.zeros((n_time, n_manifest, 1), dtype=np.float64)
    weights = np.zeros((n_time, n_manifest, 1), dtype=np.float64)
    emission_slots = np.full((n_time, n_manifest), -1, dtype=np.int64)

    for t in range(1, n_time):
        dt = times_np[t] - times_np[t - 1]
        support_start[t, :] = times_np[t - 1]
        support_end[t, :] = times_np[t]
        prev_coeffs[t, :, 0] = 0.5 * dt
        curr_coeffs[t, :, 0] = 0.5 * dt
        weights[t, :, 0] = dt
        emission_slots[t, :] = 0

    return ObservationSupportRuntime(
        anchor_times=times_np,
        manifest_names=manifest_names,
        support_kinds=["interval"] * n_manifest,
        summary_operators=["mean"] * n_manifest,
        anchor_policies=["support_end"] * n_manifest,
        observation_windows=["1d"] * n_manifest,
        support_start_times=support_start,
        support_end_times=support_end,
        interval_prev_coeffs=prev_coeffs,
        interval_curr_coeffs=curr_coeffs,
        interval_weights=weights,
        emission_slot_indices=emission_slots,
    )


def interval_mean_observations(point_obs: jnp.ndarray) -> jnp.ndarray:
    point_np = np.asarray(point_obs, dtype=np.float32)
    averaged = np.full_like(point_np, np.nan)
    averaged[1:] = 0.5 * (point_np[:-1] + point_np[1:])
    return jnp.asarray(averaged)


def peak_rss_mb() -> float:
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == "darwin":
        return rss / (1024.0 * 1024.0)
    return rss / 1024.0


In [ ]:
times = jnp.arange(T, dtype=jnp.float32)
lambda_mat = jnp.eye(N_MANIFEST, N_LATENT, dtype=jnp.float32)

point_observations = simulate_ssm(
    drift=jnp.diag(true_drift_diag),
    diffusion_chol=jnp.diag(true_diff_diag),
    lambda_mat=lambda_mat,
    manifest_chol=jnp.diag(true_obs_sd),
    t0_means=jnp.zeros(N_LATENT, dtype=jnp.float32),
    t0_chol=jnp.diag(true_t0_sd),
    times=times,
    rng_key=random.PRNGKey(SEED),
)

if USE_INTERVAL_SUPPORT:
    observations = interval_mean_observations(point_observations)
    observation_support = build_interval_mean_support(
        times,
        [f"y{i}" for i in range(N_MANIFEST)],
    )
else:
    observations = point_observations
    observation_support = None

spec = make_diagonal_spec(N_LATENT, N_MANIFEST, lambda_mat)
model = SSMModel(spec, likelihood="particle")
model.set_observation_support(observation_support)

print(f"observations shape: {observations.shape}")
print(f"interval support enabled: {USE_INTERVAL_SUPPORT}")
print(f"model likelihood backend: {model.likelihood}")
pd.DataFrame(np.asarray(observations[:5]), columns=spec.manifest_names)


In [ ]:
usage_before = resource.getrusage(resource.RUSAGE_SELF)
start = time.perf_counter()

result = fit(
    model,
    observations=observations,
    times=times,
    method="laplace_em",
    num_samples=NUM_SAMPLES,
    n_ieks_iters=N_IEKS_ITERS,
    maxiter=MAXITER,
    tol=TOL,
    seed=SEED,
)

wall_sec = time.perf_counter() - start
usage_after = resource.getrusage(resource.RUSAGE_SELF)
cpu_sec = (usage_after.ru_utime + usage_after.ru_stime) - (
    usage_before.ru_utime + usage_before.ru_stime
)
peak_rss = peak_rss_mb()

assert result.method == "laplace_em"
assert model.likelihood == "particle"
if USE_INTERVAL_SUPPORT:
    assert result.diagnostics["optimizer"] == "L-BFGS-B"

print(f"optimizer: {result.diagnostics['optimizer']}")
print(f"success: {result.diagnostics['success']}")
print(f"status: {result.diagnostics['status']}")
print(f"iterations: {result.diagnostics['n_iters']}")
print(f"function evals: {result.diagnostics['n_function_evals']}")
print(f"wall time (s): {wall_sec:.2f}")
print(f"cpu time (s): {cpu_sec:.2f}")
print(f"peak RSS (MB): {peak_rss:.1f}")
print(f"backend object: {type(result.diagnostics['likelihood_backend']).__name__}")


In [ ]:
samples = result.get_samples()

families = [
    ("drift", -jnp.abs(samples["drift_diag_free"]), true_drift_diag),
    ("diffusion_sd", samples["diffusion_diag_free"], true_diff_diag),
    ("obs_sd", samples["manifest_var_diag_free"], true_obs_sd),
]

rows = []
for family, draws, truth in families:
    means = jnp.mean(draws, axis=0)
    q05 = jnp.quantile(draws, 0.05, axis=0)
    q95 = jnp.quantile(draws, 0.95, axis=0)
    for idx in range(draws.shape[1]):
        truth_i = float(truth[idx])
        mean_i = float(means[idx])
        q05_i = float(q05[idx])
        q95_i = float(q95[idx])
        rows.append(
            {
                "family": family,
                "index": idx,
                "truth": truth_i,
                "posterior_mean": mean_i,
                "q05": q05_i,
                "q95": q95_i,
                "covered": q05_i <= truth_i <= q95_i,
                "abs_error": abs(mean_i - truth_i),
                "ci_width": q95_i - q05_i,
            }
        )

recovery_df = pd.DataFrame(rows)
family_summary = recovery_df.groupby("family").agg(
    coverage=("covered", "mean"),
    mean_abs_error=("abs_error", "mean"),
    mean_ci_width=("ci_width", "mean"),
)

display(family_summary)
recovery_df.head(12)


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for ax, family in zip(axes, ["drift", "diffusion_sd", "obs_sd"], strict=True):
    sub = recovery_df[recovery_df["family"] == family].sort_values("index")
    y = sub["posterior_mean"].to_numpy()
    yerr = np.vstack(
        [
            y - sub["q05"].to_numpy(),
            sub["q95"].to_numpy() - y,
        ]
    )
    ax.errorbar(
        sub["index"],
        y,
        yerr=yerr,
        fmt="o",
        capsize=3,
        label="posterior mean ± 90% CI",
    )
    ax.plot(sub["index"], sub["truth"], "k--", linewidth=1.5, label="truth")
    ax.set_title(family)
    ax.grid(alpha=0.3)

axes[0].legend(loc="best")
axes[-1].set_xlabel("latent / manifest index")
plt.tight_layout()


## Knobs To Turn

- If the fit is too slow, reduce `T`, `NUM_SAMPLES`, or `MAXITER` first.
- If you want the faster non-support-aware Laplace path while still avoiding Kalman, set `USE_INTERVAL_SUPPORT = False` but keep `likelihood="particle"`.
- If recovery is weak for some coordinates, increase `T` before increasing `NUM_SAMPLES`.
- If you want a harder benchmark, free a sparse set of off-diagonal drift terms instead of keeping the drift diagonal.
